### 03 · Gradiente reconstruido (histórico y simulaciones)
PCs de presión → campo (`inverse_transform`) → gradiente² (`spatial_gradient`) → 10 PCs del gradiente (PCA del gradiente real, 1985–2015). Dominio Mediterráneo occidental, igual que el 01. Para cada generador del 02A: `outputs/grad_rec/simulaciones_{nombre}/grec_XXX.parquet` (los que ya existen se saltan).

In [1]:
import os, glob

import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from bluemath_tk.datamining import PCA
from bluemath_tk.core.io import load_model

from sklearn.decomposition import PCA as PCA_
from bluemath_tk.core.operations import spatial_gradient
from joblib import Parallel, delayed

#### Load Data

In [2]:
pcs_df = pd.read_csv('outputs/pca/pcs_msl_30_6h.csv', index_col=0, parse_dates=True)
cols = list(pcs_df.columns)

pca_msl = load_model('outputs/pca/pca_msl.pkl')

era5 = xr.open_zarr('data/era5_mslp_grad_1985_2026_1deg_hourly.zarr').rename({'valid_time': 'time'})
era5 = era5.sel(time=era5.time.dt.hour.isin([0, 6, 12, 18]))
era5 = era5.sel(latitude=slice(48, 30), longitude=slice(-5, 20))                     # dominio: Mediterráneo occidental

training = slice('1985-01-01', '2015-12-31')


#### PCA del gradiente real (bluemath)

In [3]:
pca_grad = PCA(n_components=50)
pca_grad._pca = PCA_(n_components=50, svd_solver='covariance_eigh')
pca_grad.fit_transform(data=era5[['grad2']].sel(time=training), vars_to_stack=['grad2'],
                       coords_to_stack=['latitude', 'longitude'], pca_dim_for_rows='time')

cum = np.cumsum(pca_grad.explained_variance_ratio)
print(f'10 PCs: {cum[9]:.3f} | 50 PCs: {cum[49]:.3f}')

2026-11-26 16:56:07,339 - PCA - WARNING - Using PCA
2026-11-26 16:56:09,344 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method


10 PCs: 0.587 | 50 PCs: 0.872


#### PCs de presión → campo (inverse_transform) → gradiente → PCs del gradiente

In [4]:
def grad_pcs(pcs, n_gr=10, paso=8000):
    out = []
    for i in range(0, len(pcs), paso):
        p = pcs.iloc[i:i + paso][cols]
        pcs_da = xr.DataArray(p.values, dims=('time', 'n_component'),
                              coords={'time': p.index, 'n_component': np.arange(len(cols))})
        campo = pca_msl.inverse_transform(pcs_da)['msl'].transpose('time', 'latitude', 'longitude')
        grad = spatial_gradient(campo).to_dataset(name='grad2')
        out.append(pca_grad.transform(data=grad)['PCs'].values[:, :n_gr])
    return pd.DataFrame(np.concatenate(out), index=pcs.index, columns=[f'grec_PC{k + 1}' for k in range(n_gr)])

#### Histórico

In [5]:
grec_hist = grad_pcs(pcs_df)

#### Comprobación: reconstruido vs real en training
grec_real = pca_grad.transform(data=era5[['grad2']].sel(time=training))['PCs'].values[:, :10]
t_train = grec_hist.loc[training].index
print(pd.Series([np.corrcoef(grec_real[:, k], grec_hist.loc[t_train].values[:, k])[0, 1] for k in range(10)],
                index=grec_hist.columns).round(2))

2026-11-26 16:56:12,061 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 16:56:14,507 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 16:56:16,936 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 16:56:19,198 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 16:56:21,639 - PCA - WARNING - Using 450 out of 450 available variables 
If this is originated by using few times, please check 'nan_threshold_to_drop' parameter in fit method
2026-11-26 16:56:25,481 - PCA - WARNING - Using 450 out of 450 av

grec_PC1     0.99
grec_PC2     0.97
grec_PC3     0.97
grec_PC4     0.96
grec_PC5     0.93
grec_PC6     0.95
grec_PC7     0.91
grec_PC8     0.90
grec_PC9     0.83
grec_PC10    0.85
dtype: float64


In [6]:
os.makedirs('outputs/grad_rec', exist_ok=True)
grec_hist.to_csv('outputs/grad_rec/pcs_grad_rec_6h.csv', float_format='%.5f')
pca_grad.save_model('outputs/grad_rec/pca_grad.pkl')

2026-11-26 16:57:03,442 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!


#### Simulaciones de los generadores del 02A

In [7]:
GENERADORES = ['h1b', 'h1bb24', 'h1']            # en este orden: los de bootstrap acaban antes en el 02A

def procesar(f, out_dir):
    f_out = os.path.join(out_dir, os.path.basename(f).replace('sim_', 'grec_'))
    if not os.path.exists(f_out):
        grad_pcs(pd.read_parquet(f)).astype('float32').to_parquet(f_out)

for nombre in GENERADORES:
    sim_files = sorted(glob.glob(f'outputs/var_6h/simulaciones_{nombre}/sim_*.parquet'))
    out_dir = f'outputs/grad_rec/simulaciones_{nombre}'
    os.makedirs(out_dir, exist_ok=True)
    print(nombre, len(sim_files))
    _ = Parallel(n_jobs=os.cpu_count() - 2, verbose=5)(delayed(procesar)(f, out_dir) for f in sim_files)

[Parallel(n_jobs=22)]: Using backend LokyBackend with 22 concurrent workers.
2026-11-26 16:57:03,604 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 16:57:03,645 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!


h1b 100


2026-11-26 16:57:04,078 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 16:57:04,084 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 16:57:04,096 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 16:57:04,099 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 16:57:04,114 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 16:57:04,119 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 16:57:04,132 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 16:57:04,136 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 16:57:04,150 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26

h1bb24 100


2026-11-26 17:07:27,047 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:07:27,056 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:07:27,056 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:07:27,071 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:07:27,076 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:07:27,088 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:07:27,091 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:07:27,104 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:07:27,110 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26

h1 100


2026-11-26 17:16:43,808 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:16:43,816 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:16:43,831 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:16:43,834 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:16:43,847 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:16:43,851 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:16:43,865 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:16:43,871 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26 17:16:43,883 - PCA - WARNING - Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
2026-11-26